# NullVector LangGraph QA Agent — PostgreSQL Backend

This notebook builds a generative QA agent over a NullVector corpus stored in PostgreSQL.

**Prerequisite:** Run `03_nullvector_postgres_unified.ipynb` first to populate the Postgres database with acquisition, tree, and retrieval artifacts. This notebook loads those artifacts by run ID.

**Requirements:**
- A reachable PostgreSQL database with NullVector artifacts from notebook 03
- `OPENROUTER_API_KEY` environment variable for the LLM agent
- Optional: `NULLVECTOR_POSTGRES_CONNINFO` (defaults to `postgresql://REDACTED_DB_CRED@localhost:5432/nullvector`)

## Section 1 — Dependencies

Install the external LangGraph and LangChain packages. These are not NullVector dependencies — they are optional agent-framework integrations.

In [ ]:
!uv pip install -q "langgraph>=0.2" "langchain-openai>=0.1" "langchain-core>=0.2"

## Section 2 — Load Corpus from PostgreSQL

We reuse the same run IDs and PostgreSQL config from notebook 03 to load the Markdown corpus and node cards.

In [ ]:
from __future__ import annotations

import os
import textwrap
from collections import Counter
from typing import Annotated, TypedDict

from nullvector.domain import NodeCard
from nullvector.domain.retrieval import RetrievalUnitType
from nullvector.retrieval import QueryPlanner, RetrievalRanker, RetrievalService, load_retrieval_corpus
from nullvector.storage import PostgresStorageConfig, build_document_store, build_postgres_artifact_ref

POSTGRES_CONNINFO = os.environ.get(
    "NULLVECTOR_POSTGRES_CONNINFO",
    "postgresql://REDACTED_DB_CRED@localhost:5432/nullvector",
)
POSTGRES_SCHEMA = os.environ.get("NULLVECTOR_POSTGRES_SCHEMA", "public")
OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY"]
AGENT_MODEL = os.environ.get("NULLVECTOR_LLM_MODEL", "openrouter/openai/gpt-4.1-mini")

pg_config = PostgresStorageConfig(conninfo=POSTGRES_CONNINFO, schema=POSTGRES_SCHEMA)
store = build_document_store(pg_config)

# Run IDs matching notebook 03
MARKDOWN_RETRIEVAL_RUN_ID = "cookbook-md-retrieval"
MARKDOWN_TREE_RUN_ID = "cookbook-md-tree"

# Discover document_id from the retrieval run record
run_record = store.read_json_artifact(
    build_postgres_artifact_ref(
        run_type="retrieval",
        run_id=MARKDOWN_RETRIEVAL_RUN_ID,
        document_id="*",
        artifact_path="run-index.json",
    )
)
DOCUMENT_ID = run_record["document_id"]

CORPUS_PATH = build_postgres_artifact_ref(
    run_type="retrieval",
    run_id=MARKDOWN_RETRIEVAL_RUN_ID,
    document_id=DOCUMENT_ID,
    artifact_path="corpus.json",
)
corpus = load_retrieval_corpus(CORPUS_PATH, storage=pg_config)

# Load node cards for title display
TREE_MANIFEST_PATH = build_postgres_artifact_ref(
    run_type="tree",
    run_id=MARKDOWN_TREE_RUN_ID,
    document_id=DOCUMENT_ID,
    artifact_path="manifest.json",
)
tree_manifest = store.read_json_artifact(TREE_MANIFEST_PATH)
node_cards_payload = store.read_json_artifact(tree_manifest["node_cards_path"])
node_cards = tuple(NodeCard.model_validate(item) for item in node_cards_payload)
titles_by_id = {card.node_id: card.title for card in node_cards}

counts = Counter(unit.unit_type.value for unit in corpus.units)
print(f"Loaded corpus: {len(corpus.units)} units from document {DOCUMENT_ID}")
print(f"Unit types: {dict(sorted(counts.items()))}")
print(f"Node titles: {[card.title for card in node_cards]}")

## Section 3 — NullVector Retrieval Tools

Two tools expose NullVector retrieval to the LangGraph agent:
- `search_document` — runs `RetrievalService.search()` and formats results
- `get_page_text` — returns the full text of a specific logical page

In [ ]:
from langchain_core.tools import tool

planner = QueryPlanner()
ranker = RetrievalRanker()
retrieval_service = RetrievalService(planner, ranker, storage=pg_config)


@tool
def search_document(query: str, limit: int = 5) -> str:
    """Search the NullVector document corpus for relevant content.

    Use this tool to find evidence before answering questions.
    Returns ranked retrieval hits with page numbers and excerpts.
    """
    hits = retrieval_service.search(corpus=corpus, query=query, limit=limit)
    if not hits:
        return "No results found for this query."
    lines: list[str] = []
    for i, hit in enumerate(hits, 1):
        unit = hit.unit
        excerpt = textwrap.shorten(unit.text, width=200, placeholder="...")
        node_title = titles_by_id.get(unit.node_id or "", "")
        lines.append(
            f"[{i}] score={hit.score:.3f} | "
            f"type={unit.unit_type.value} | "
            f"pages={unit.page_span.start_page}-{unit.page_span.end_page} | "
            f"node={node_title!r}\n    {excerpt}"
        )
    return "\n\n".join(lines)


@tool
def get_page_text(page_number: int) -> str:
    """Get the complete text of a specific logical page (0-indexed).

    Use this after search_document identifies relevant pages.
    """
    for unit in corpus.units:
        if (
            unit.unit_type is RetrievalUnitType.PAGE_TEXT
            and unit.page_span.start_page == page_number
        ):
            return unit.text
    return f"Page {page_number} not found in corpus."


TOOLS = [search_document, get_page_text]
TOOL_MAP = {t.name: t for t in TOOLS}
print(f"Registered tools: {list(TOOL_MAP.keys())}")

## Section 4 — LangGraph Agent Construction

Standard ReAct agent: the LLM decides whether to call tools or produce a final answer. Tool dispatch uses a manual `execute_tools` node with error handling rather than the prebuilt `ToolNode`.

In [ ]:
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages

SYSTEM_PROMPT = (
    "You are a precise document QA assistant backed by NullVector. "
    "Always use search_document to find evidence before answering. "
    "Cite page numbers when referencing content. "
    "If the corpus does not contain enough evidence, say so clearly."
)


class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


llm = ChatOpenAI(
    model=AGENT_MODEL.removeprefix("openrouter/"),
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
).bind_tools(TOOLS)


def call_model(state: AgentState) -> dict[str, list[BaseMessage]]:
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    return {"messages": [llm.invoke(messages)]}


def execute_tools(state: AgentState) -> dict[str, list[ToolMessage]]:
    last: AIMessage = state["messages"][-1]
    results: list[ToolMessage] = []
    for tc in last.tool_calls:
        fn = TOOL_MAP.get(tc["name"])
        try:
            output = fn.invoke(tc["args"]) if fn else f"Unknown tool: {tc['name']}"
        except Exception as exc:
            output = f"Tool error: {exc}"
        results.append(ToolMessage(content=str(output), tool_call_id=tc["id"], name=tc["name"]))
    return {"messages": results}


def should_continue(state: AgentState) -> str:
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    return END


graph = StateGraph(AgentState)
graph.add_node("agent", call_model)
graph.add_node("tools", execute_tools)
graph.set_entry_point("agent")
graph.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
graph.add_edge("tools", "agent")

memory = MemorySaver()
app = graph.compile(checkpointer=memory)

print("LangGraph agent compiled successfully")

In [ ]:
def ask(question: str, *, thread_id: str = "main", verbose: bool = False) -> str:
    """Submit a question to the agent. Same thread_id = multi-turn context."""
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke({"messages": [HumanMessage(content=question)]}, config)
    messages = result["messages"]

    if verbose:
        print("=" * 60)
        print("TRACE")
        print("=" * 60)
        for msg in messages:
            role = type(msg).__name__
            content = textwrap.shorten(str(msg.content), width=120, placeholder="...")
            print(f"  [{role}] {content}")
        print("=" * 60)

    last = messages[-1]
    return last.content if isinstance(last, AIMessage) else str(last)

## Section 5 — Single-Turn QA Demo

The agent searches the corpus, retrieves evidence, and produces a grounded answer.

In [ ]:
answer = ask(
    "What do the litigation policies say about case deadlines?",
    thread_id="single-turn",
    verbose=True,
)
print()
print("ANSWER:")
print(answer)

## Section 6 — Multi-Turn Conversation

Using the same `thread_id` preserves conversation context across turns. The agent can reference earlier answers and retrieve additional evidence on follow-ups.

In [ ]:
THREAD = "multi-turn-demo"

print("Q1:", "What sections does the operating handbook cover?")
a1 = ask("What sections does the operating handbook cover?", thread_id=THREAD)
print("A1:", a1)
print()

print("Q2:", "Tell me more about the controls checklist from that handbook.")
a2 = ask("Tell me more about the controls checklist from that handbook.", thread_id=THREAD)
print("A2:", a2)
print()

print("Q3:", "How does the customer support section relate to the controls?")
a3 = ask("How does the customer support section relate to the controls?", thread_id=THREAD)
print("A3:", a3)

## Section 7 — Streaming Output

Stream agent responses to see tool calls and intermediate steps as they happen.

In [ ]:
config = {"configurable": {"thread_id": "streaming-demo"}}
payload = {"messages": [HumanMessage(content="What are the revenue policies about?")]}

print("=" * 60)
print("STREAMING")
print("=" * 60)
for event in app.stream(payload, config, stream_mode="values"):
    last = event["messages"][-1]
    role = type(last).__name__
    if isinstance(last, AIMessage) and last.tool_calls:
        for tc in last.tool_calls:
            print(f"  [{role}] calling {tc['name']}({tc['args']})")
    elif isinstance(last, ToolMessage):
        excerpt = textwrap.shorten(last.content, width=100, placeholder="...")
        print(f"  [{role}] {last.name} -> {excerpt}")
    elif isinstance(last, AIMessage):
        print(f"\n  [{role}] FINAL ANSWER:")
        print(f"  {last.content}")
print("=" * 60)

## Notes

- This notebook requires `OPENROUTER_API_KEY` — there is no noop fallback because the LangGraph agent needs a live LLM.
- The corpus is loaded from PostgreSQL using the same run IDs as notebook 03. If you used different run IDs, update `MARKDOWN_RETRIEVAL_RUN_ID` and `MARKDOWN_TREE_RUN_ID`.
- Tool definitions close over the `corpus` and `retrieval_service` objects loaded in Section 2.
- `MemorySaver` provides in-process multi-turn memory scoped by `thread_id`. For persistent memory across sessions, replace with a LangGraph persistence backend.